In [2]:
import os
import shutil
import random
from glob import glob

# --- 設定 ---
SOURCE_DIR = "roboflow_1000_側拍"
TARGET_BASE_DIR = "roboflow_1000_側拍_split"
CLASSES_FILE = os.path.join(SOURCE_DIR, "classes.txt")

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

SEED = 100
random.seed(SEED)

IMAGE_EXT = ".jpg"

# --- 1️⃣ 收集每張圖片的「所有類別」 ---
print("🔍 正在分析多類別分佈（YOLO正確模式）...")

image_files = glob(os.path.join(SOURCE_DIR, "*" + IMAGE_EXT))
image_classes = {}

for img_path in image_files:
    name = os.path.basename(img_path).replace(IMAGE_EXT, "")
    txt_path = os.path.join(SOURCE_DIR, name + ".txt")

    classes = set()

    if os.path.exists(txt_path):
        with open(txt_path, 'r') as f:
            for line in f:
                if line.strip():
                    cls = line.split()[0]
                    classes.add(cls)

    # 沒有標註 → 當成 empty 類
    if not classes:
        classes.add("empty")

    image_classes[name] = classes

# --- 2️⃣ Stratified Split（多類別） ---
print("⚖️ 正在進行分層切分...")

train_names, val_names, test_names = set(), set(), set()

all_names = list(image_classes.keys())
random.shuffle(all_names)

n = len(all_names)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train_names = set(all_names[:n_train])
val_names = set(all_names[n_train:n_train+n_val])
test_names = set(all_names[n_train+n_val:])

print(f"✅ 切分完成")
print(f"Train: {len(train_names)}")
print(f"Val:   {len(val_names)}")
print(f"Test:  {len(test_names)}")

# --- 4️⃣ 建立資料夾 ---
def setup_dirs(split):
    base = os.path.join(TARGET_BASE_DIR, split)
    if os.path.exists(base):
        shutil.rmtree(base)
    os.makedirs(os.path.join(base, "images"), exist_ok=True)
    os.makedirs(os.path.join(base, "labels"), exist_ok=True)

splits = {
    "train": train_names,
    "val": val_names,
    "test": test_names
}

empty_txt_count = 0

# --- 5️⃣ 複製資料 ---
for split, names in splits.items():
    setup_dirs(split)

    img_dir = os.path.join(TARGET_BASE_DIR, split, "images")
    lbl_dir = os.path.join(TARGET_BASE_DIR, split, "labels")

    print(f"🚀 複製 {split} 資料...")

    for name in names:
        src_img = os.path.join(SOURCE_DIR, name + IMAGE_EXT)
        dst_img = os.path.join(img_dir, name + IMAGE_EXT)

        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)

        src_lbl = os.path.join(SOURCE_DIR, name + ".txt")
        dst_lbl = os.path.join(lbl_dir, name + ".txt")

        if os.path.exists(src_lbl) and os.path.getsize(src_lbl) > 0:
            shutil.copy(src_lbl, dst_lbl)
        else:
            open(dst_lbl, 'w').close()
            empty_txt_count += 1

# --- 6️⃣ 產生 data.yaml ---
print("\n📝 產生 data.yaml...")

class_names = []
if os.path.exists(CLASSES_FILE):
    with open(CLASSES_FILE, 'r', encoding='utf-8') as f:
        class_names = [line.strip() for line in f if line.strip()]

yaml_content = f"""# YOLO Dataset
path: {os.path.abspath(TARGET_BASE_DIR)}
train: train/images
val: val/images
test: test/images

nc: {len(class_names)}
names: {class_names}
"""

with open(os.path.join(TARGET_BASE_DIR, "data.yaml"), "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"✅ 完成！空標註檔數量: {empty_txt_count}")

🔍 正在分析多類別分佈（YOLO正確模式）...
⚖️ 正在進行分層切分...
✅ 切分完成
Train: 800
Val:   100
Test:  100
🚀 複製 train 資料...
🚀 複製 val 資料...
🚀 複製 test 資料...

📝 產生 data.yaml...
✅ 完成！空標註檔數量: 0


In [ ]:
#分析切分後的類別分佈
import os
import yaml
import pandas as pd

# --- 設定路徑 ---
YOLO_DATA_DIR = 'roboflow_1000_側拍_split'
DATA_YAML_PATH = os.path.join(YOLO_DATA_DIR, 'data.yaml')

# --- 函數：讀取並統計標註 ---
def count_labels(label_dir, class_names):
    """
    遍歷指定資料夾內的 .txt 標註檔，並統計:
    1. 每個類別的物件數量
    2. 純背景圖片 (空標註檔) 的數量
    """
    counts = {name: 0 for name in class_names}
    total_files = 0
    total_objects = 0
    background_files = 0 
    
    if not os.path.exists(label_dir):
        print(f"⚠️ 警告：找不到路徑 {label_dir}。跳過此分割。")
        return counts, 0, 0, 0

    for filename in os.listdir(label_dir):
        if filename.endswith('.txt'):
            file_path = os.path.join(label_dir, filename)
            total_files += 1
            
            try:
                # 檢查是否為空檔案 (0 bytes)
                if os.path.getsize(file_path) == 0:
                    background_files += 1
                    continue

                has_objects = False
                with open(file_path, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        
                        has_objects = True 
                        
                        try:
                            # YOLO 格式: <class_id> <x> <y> <w> <h>
                            class_index = int(line.split()[0])
                            
                            if 0 <= class_index < len(class_names):
                                class_name = class_names[class_index]
                                counts[class_name] += 1
                                total_objects += 1
                        except (ValueError, IndexError):
                            continue 
                
                if not has_objects:
                    background_files += 1
                                    
            except Exception as e:
                print(f"❌ 錯誤：讀取標註檔 {filename} 失敗: {e}")
                continue

    return counts, total_files, total_objects, background_files


# --- 主程式流程 ---
def analyze_split_distribution():
    # 1. 載入類別名稱
    try:
        with open(DATA_YAML_PATH, 'r', encoding='utf-8') as f:
            data_yaml = yaml.safe_load(f)
        
        class_names = data_yaml.get('names')
        if not class_names:
            raise ValueError("在 data.yaml 中找不到 'names' 列表。")
        
        print(f"✅ 成功讀取 {len(class_names)} 個類別: {class_names}")
    
    except FileNotFoundError:
        print(f"❌ 錯誤：找不到 data.yaml 檔案: {DATA_YAML_PATH}")
        return
    except Exception as e:
        print(f"❌ 錯誤：解析 data.yaml 失敗: {e}")
        return

    # 2. 統計各個分割集
    splits = ['train', 'val', 'test']
    results = {}

    for s in splits:
        label_dir = os.path.join(YOLO_DATA_DIR, s, 'labels')
        counts, files, objects, bg = count_labels(label_dir, class_names)
        results[s] = {
            'counts': counts,
            'files': files,
            'objects': objects,
            'bg': bg
        }

    # 3. 整理資料夾物件分佈表格
    df_data = {
        'Train Objects': results['train']['counts'],
        'Val Objects': results['val']['counts'],
        'Test Objects': results['test']['counts']
    }
    df = pd.DataFrame(df_data).T 
    df['Total Objects'] = df.sum(axis=1)
    
    # 4. 輸出總結
    print("\n" + "="*60)
    print("--- 訓練 / 驗證 / 測試資料集統計總結 ---")
    print("="*60)
    
    for s in splits:
        icon = "📘" if s == 'train' else ("📙" if s == 'val' else "📗")
        name = s.capitalize()
        res = results[s]
        print(f"{icon} {name} Set:")
        print(f"   - 總圖片數:      {res['files']}")
        print(f"   - 有物件圖片:    {res['files'] - res['bg']}")
        print(f"   - 純背景圖片:    {res['bg']} (負樣本)")
        print(f"   - 標註物件總數:  {res['objects']}")
        print("-" * 40)
    
    # 輸出詳細的類別分佈表
    print("\n📦 類別物件數量分佈 (Object Counts)")
    print(df.to_markdown(numalign="left", stralign="left"))

    # 5. 檢查分佈平衡性 (比例分析)
    print("\n💡 平衡性檢查 (以 Train 為基準的比例)")
    print(f"{'Class Name':<15} | {'Train:Val':<10} | {'Train:Test':<10}")
    print("-" * 45)

    def get_ratio_str(train_val, target_val):
        if target_val == 0:
            return "Inf" if train_val > 0 else "0.0"
        return f"{train_val / target_val:.1f}"

    # 背景圖比例
    bg_v = get_ratio_str(results['train']['bg'], results['val']['bg'])
    bg_t = get_ratio_str(results['train']['bg'], results['test']['bg'])
    print(f"{'[Background]':<15} | {bg_v:<10} | {bg_t:<10}")

    for class_name in class_names:
        tr_c = results['train']['counts'].get(class_name, 0)
        va_c = results['val']['counts'].get(class_name, 0)
        te_c = results['test']['counts'].get(class_name, 0)
        
        ratio_v = get_ratio_str(tr_c, va_c)
        ratio_t = get_ratio_str(tr_c, te_c)

        print(f"{class_name:<15} | {ratio_v:<10} | {ratio_t:<10}")
    
    print("\n(註：若比例設定為 8:1:1，建議比例約為 8.0)")     
    print("="*60)


if __name__ == "__main__":
    analyze_split_distribution()

In [ ]:
# 計算尺寸統計
import os
import yaml
import glob
from PIL import Image
import pandas as pd
import numpy as np
import sys

# --- 參數設定 ---
YOLO_DATA_DIR = 'roboflow_1000_sample_split'
DATA_YAML_PATH = os.path.join(YOLO_DATA_DIR, 'data.yaml')

def load_config(yaml_path):
    """載入 data.yaml 並進行基本檢查"""
    try:
        with open(yaml_path, 'r', encoding='utf-8') as f:
            data_yaml = yaml.safe_load(f)
        
        if not isinstance(data_yaml, dict):
            print(f"❌ 錯誤：data.yaml 格式不正確，應該是字典格式，但讀取到 {type(data_yaml)}")
            return None
            
        return data_yaml
    except FileNotFoundError:
        print(f"❌ 錯誤：找不到 data.yaml 檔案於 {yaml_path}")
        return None
    except Exception as e:
        print(f"❌ 錯誤：無法解析 {yaml_path}。詳細: {e}")
        return None

def resolve_path(path_in_yaml, yaml_file_path):
    """
    智慧路徑解析：嘗試多種可能性來找到真實路徑
    1. 檢查是否為絕對路徑或相對於當前工作目錄存在的路徑
    2. 檢查是否相對於 data.yaml 檔案所在位置
    """
    if not path_in_yaml:
        return None

    # 1. 直接檢查 (適用於絕對路徑 或 相對於執行腳本的路徑)
    if os.path.exists(path_in_yaml):
        return path_in_yaml
    
    # 2. 相對於 yaml 檔案的位置 (YOLO 標準做法)
    yaml_dir = os.path.dirname(os.path.abspath(yaml_file_path))
    path_relative_to_yaml = os.path.join(yaml_dir, path_in_yaml)
    if os.path.exists(path_relative_to_yaml):
        return path_relative_to_yaml
        
    return None

def get_data_paths(config):
    """獲取圖片與標註檔的配對路徑"""
    paths = []
    
    for split in ['train', 'val', 'test']:
        images_rel = config.get(split)
        
        # 使用智慧路徑解析
        images_dir = resolve_path(images_rel, DATA_YAML_PATH)
        
        if images_dir:
            # 假設 labels 資料夾在 images 資料夾同層級的 ../labels
            # 這是 YOLO 的標準結構
            parent_dir = os.path.dirname(images_dir)
            labels_dir = os.path.join(parent_dir, 'labels')
            
            # 再次確認路徑存在
            if not os.path.exists(labels_dir):
                # 嘗試另一種結構：如果 images_dir 已經包含 images 字樣，直接替換
                if 'images' in images_dir:
                    labels_dir = images_dir.replace('images', 'labels')
            
            if os.path.exists(images_dir):
                # 搜尋圖片
                found_images = glob.glob(os.path.join(images_dir, '*'))
                valid_images = [img for img in found_images if img.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                
                if not valid_images:
                    print(f"⚠️ 警告: 在 {images_dir} 中找不到任何圖片。")
                    continue
                    
                for img_path in valid_images:
                    paths.append((img_path, labels_dir))
            else:
                print(f"⚠️ 警告: 無法定位 {split} 資料夾: {images_rel}")
        else:
            if images_rel:
                print(f"⚠️ 警告: 設定檔中的路徑無效: {images_rel}")
                
    return paths

def analyze_simple_resolution():
    
    # 檢查依賴
    try:
        import pandas as pd
        import numpy as np
    except ImportError:
        print("❌ 錯誤：需要安裝函式庫。請執行：pip install Pillow pandas numpy")
        sys.exit(1)
            
    print(f"📂 正在讀取設定: {DATA_YAML_PATH}")
    config = load_config(DATA_YAML_PATH)
    if config is None:
        return
        
    class_names = config.get('names', [])
    print(f"📋 偵測到類別: {class_names}")
    
    data_paths = get_data_paths(config)
    
    if not data_paths:
        print("❌ 未找到任何有效的圖片路徑。請檢查 yolo_data/data.yaml 的內容。")
        return

    print(f"✅ 成功載入 {len(data_paths)} 張圖片，開始分析...")

    results_list = []

    for img_path, labels_dir in data_paths:
        file_basename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(labels_dir, file_basename + '.txt')

        # 1. 讀取圖片尺寸
        try:
            with Image.open(img_path) as img:
                img_width, img_height = img.size
        except Exception:
            continue
        
        # 2. 讀取標註並轉換為像素尺寸
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    try:
                        parts = line.strip().split()
                        if len(parts) < 5: continue
                        
                        class_id = int(parts[0])
                        norm_w = float(parts[3])
                        norm_h = float(parts[4])
                        
                        bbox_width_px = norm_w * img_width
                        bbox_height_px = norm_h * img_height
                        bbox_area_px = bbox_width_px * bbox_height_px
                        
                        class_name = class_names[class_id] if 0 <= class_id < len(class_names) else f'Class_{class_id}'
                        
                        results_list.append({
                            'Class': class_name,
                            'Photo_Width_px': img_width,
                            'Photo_Height_px': img_height,
                            'BBox_Width_px': bbox_width_px,
                            'BBox_Height_px': bbox_height_px,
                            'BBox_Area_px': bbox_area_px,
                        })
                    except Exception:
                        continue

    if not results_list:
        print("❌ 雖然找到了圖片，但在對應的標註檔中沒有找到任何物件。")
        return

    df = pd.DataFrame(results_list)

    # --- 輸出統計結果 ---
    print("\n" + "="*50)
    print("--- 資料集解析度與標註尺寸統計 ---")
    print("="*50)

    # 1. 圖片解析度
    photo_df = df[['Photo_Width_px', 'Photo_Height_px']].drop_duplicates()
    photo_stats = photo_df.agg(['mean', 'median', 'min', 'max']).T
    
    print("\n🖼️ 圖片原始解析度 (Pixel)")
    print(photo_stats.to_markdown(numalign="left", stralign="left", floatfmt=".1f"))

    # 2. 標註框尺寸
    bbox_stats = df.groupby('Class')[['BBox_Width_px', 'BBox_Height_px', 'BBox_Area_px']].agg(['count', 'mean', 'median', 'min', 'max']).fillna(0)
    
    print("\n📦 標註框像素尺寸統計 (Pixel)")
    
    for class_name in bbox_stats.index:
        stats = bbox_stats.loc[class_name]
        count = int(stats['BBox_Width_px']['count'])
        
        print(f"\n--- 類別: {class_name} (總數: {count}) ---")
        
        output_data = {
            'Metric': ['Width', 'Height', 'Area'],
            'Mean': [stats['BBox_Width_px']['mean'], stats['BBox_Height_px']['mean'], stats['BBox_Area_px']['mean']],
            'Median': [stats['BBox_Width_px']['median'], stats['BBox_Height_px']['median'], stats['BBox_Area_px']['median']],
            'Min': [stats['BBox_Width_px']['min'], stats['BBox_Height_px']['min'], stats['BBox_Area_px']['min']],
            'Max': [stats['BBox_Width_px']['max'], stats['BBox_Height_px']['max'], stats['BBox_Area_px']['max']]
        }
        print(pd.DataFrame(output_data).to_markdown(index=False, floatfmt=".1f"))
        
    print("\n" + "="*50)
    print("統計完成。")

if __name__ == "__main__":
    analyze_simple_resolution()

In [1]:
# 清除舊的 .cache 和 .npy 檔案
import os
import glob

# 設定您的資料集路徑
dataset_dir = 'roboflow_1000_側拍_split'  # 請確認這是您的資料集根目錄

print("🔍 正在搜尋並刪除舊的 .cache 及 .npy 檔案...")

# 搜尋 train 和 val 資料夾下的 .cache 和 .npy
cache_files = glob.glob(os.path.join(dataset_dir, '**', '*.cache'), recursive=True) + glob.glob(os.path.join(dataset_dir, '**', '*.npy'), recursive=True)

if cache_files:
    for f in cache_files:
        try:
            os.remove(f)
            print(f"🗑️ 已刪除: {f}")
        except Exception as e:
            print(f"❌ 無法刪除 {f}: {e}")
    print("\n✅ 舊快取已清除，下次訓練時會重新掃描資料集。")
else:
    print("✅ 未發現舊的 .cache 檔案。")

🔍 正在搜尋並刪除舊的 .cache 及 .npy 檔案...
🗑️ 已刪除: roboflow_1000_側拍_split\train\labels.cache
🗑️ 已刪除: roboflow_1000_側拍_split\val\labels.cache

✅ 舊快取已清除，下次訓練時會重新掃描資料集。


In [1]:
#yolo訓練程式碼 (Stage 1)
import os, torch, gc, sys, logging, warnings
from ultralytics import YOLO
from agent_tools import error_logger, yolo_utils

# --- 🌟 初始化：環境優化與自定義模組 ---
yolo_utils.register_yolo_modules()
warnings.filterwarnings("ignore", message=".*deterministic.*")
warnings.filterwarnings("ignore", category=UserWarning)
try:
    torch.use_deterministic_algorithms(False)
except Exception: pass

# --- 🛡️ 強制恢復日誌顯示 ---
logging.getLogger("ultralytics").setLevel(logging.INFO) # 🚀 強制重置為 INFO
os.environ["YOLO_VERBOSE"] = "True"                  # 🚀 強制環境變數為 True

# --- 🛡️ 環境隔離 ---
sys.argv = [sys.argv[0]]

if __name__ == '__main__':
    # 1. 顯存與變數清理
    if 'model' in globals(): del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        DEVICE_ID = 0
        print(f"🔥 使用 GPU: {torch.cuda.get_device_name(0)}")
    else:
        DEVICE_ID = 'cpu'

    # 2. 參數設定
    MODEL_CFG = "yolo11-strawberry-p2-cbam-s.yaml" 
    PRETRAINED_WEIGHTS = "yolo11s.pt"
    DATA_YAML_PATH = "roboflow_1000_側拍_split/data.yaml"

    try:
        # 3. 建立模型
        model = YOLO(MODEL_CFG).load(PRETRAINED_WEIGHTS)
        print(f"\n🚀 啟動一階段訓練：{MODEL_CFG}\n")

        # 4. 啟動訓練
        model.train(
            data=DATA_YAML_PATH,
            epochs=300,
            batch=32,
            imgsz=640,
            device=DEVICE_ID,
            workers=4,
            optimizer='AdamW',
            lr0=0.001,
            weight_decay=0.001,
            cos_lr=True,
            patience=30,
            box=15.0,
            cls=0.8,
            multi_scale=False, 
            plots=True,       
            verbose=True,     
            # === 數據增強 ===
            mosaic=1.0,
            mixup=0.2,
            copy_paste=0.02,
            scale=0.5,
            degrees=10.0,
            fliplr=0.5,
            hsv_h=0.015,
            hsv_s=0.7,
            hsv_v=0.4,
            warmup_epochs=3.0,
            close_mosaic=30
        )
        print("\n✨ 一階段基礎訓練順利結束！")
    except Exception as e:
        error_logger.log_error(e, context="YOLO 一階段基礎訓練")
        raise

🔥 使用 GPU: NVIDIA GeForce RTX 4060 Ti
WARNING no model scale passed. Assuming scale='s'.
Transferred 297/605 items from pretrained weights

🚀 啟動一階段訓練：yolo11-strawberry-p2-cbam-s.yaml

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.10.19 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=15.0, cache=False, cfg=None, classes=None, close_mosaic=30, cls=0.8, compile=False, conf=None, copy_paste=0.02, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=roboflow_1000__split/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, i

In [2]:
#🎯 二階段微調 (Fine-tuning)
import os, glob, torch, gc, sys, logging, warnings
from ultralytics import YOLO
from agent_tools import error_logger, yolo_utils

# --- 🌟 初始化：註冊自定義模組 ---
yolo_utils.register_yolo_modules()
warnings.filterwarnings("ignore", message=".*deterministic.*")
warnings.filterwarnings("ignore", category=UserWarning)

# --- 🛡️ 環境隔離 ---
sys.argv = [sys.argv[0]] 

if __name__ == '__main__':
    # 1. 自動尋找最新的一階段權重
    train_dirs = glob.glob(os.path.join("runs", "detect", "train*"))
    primary_train_dirs = [d for d in train_dirs if "_2" not in d]
    
    if not primary_train_dirs:
        print("❌ 找不到一階段訓練資料夾。")
    else:
        latest_primary_dir = max(primary_train_dirs, key=os.path.getmtime)
        base_name = os.path.basename(latest_primary_dir)
        stage2_name = f"{base_name}_2"
        best_weights = os.path.join(latest_primary_dir, "weights", "best.pt")

        if not os.path.exists(best_weights):
            print(f"⚠️ 找不到權重檔：{best_weights}")
        else:
            print(f"🚀 啟動二階段微調：{base_name} -> {stage2_name}")
            
            # 2. 顯存清理
            if 'model_stage2' in globals(): del model_stage2
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            
            try:
                # 3. 載入模型
                model_stage2 = YOLO(best_weights)
                DEVICE_ID = 0 if torch.cuda.is_available() else 'cpu'
                DATA_YAML_PATH = "roboflow_1000_側拍_split/data.yaml"

                # 4. 執行低學習率微調 (恢復日誌輸出)
                model_stage2.train(
                    data=DATA_YAML_PATH,
                    epochs=50,
                    batch=4,
                    imgsz=640,
                    device=DEVICE_ID,
                    name=stage2_name,
                    optimizer='AdamW',
                    lr0=0.0001,           # 學習率降低 10 倍
                    freeze=10,            # 鎖定部分層
                    multi_scale=False,
                    plots=True,           # 恢復圖表
                    verbose=True,         # 恢復詳細輸出
                    # 高強度數據增強
                    degrees=15.0,
                    translate=0.2,
                    perspective=0.001,
                    shear=2.0,
                    mosaic=1.0,
                    mixup=0.15,
                    copy_paste=0.1,
                    close_mosaic=10
                )
                print(f"\n✨ 二階段微調圓滿完成！權重已存入 runs/detect/{stage2_name}")
            except Exception as e:
                error_logger.log_error(e, context="YOLO 二階段微調")
                raise

In [ ]:
#自動關機指令
import os
import time

print("訓練完成！將於 5 分鐘後自動關機...")
time.sleep(2)  # 讓程式在原地「發呆」2 秒，讓你有時間讀完上面那句話
# 設定倒數秒數 (例如 300 秒 = 5 分鐘)
countdown_seconds = 300 

print(f"系統將在 {countdown_seconds // 60} 分鐘後自動關機。")
print("如果想取消關機並檢查結果，請在下方輸入 'n' 並按 Enter：")

# 使用帶有超時功能的輸入檢查 (或簡單的輸入)
try:
    # 在 Windows 環境下，這是一個簡單的阻斷式檢查
    user_input = input(f"按下 'n' 取消關機，直接按 Enter 或等待則繼續執行：").strip().lower()
    
    if user_input == 'n':
        print("\n❌ 已取消自動關機排程。你可以開始檢查 runs/train/exp 中的結果了！")
    else:
        print(f"\n🚀 正在執行關機指令... 倒數 {countdown_seconds} 秒。")
        # /s: 關機, /t: 倒數秒數, /f: 強制關閉執行中的程式
        os.system(f"shutdown /s /t {countdown_seconds} /f")
        print("提示：若後悔了，請在 CMD 輸入 'shutdown -a' 取消。")

except Exception as e:
    # 萬一輸入過程出錯，安全起見預設不關機
    print(f"確認過程發生錯誤: {e}，取消自動關機。")